# 01 · Data: download, cache, structural checks

Downloads the three parquet shards (about 1.15 GB) to Drive once, caches each row as arrays, then re-runs the
structural checks from the October pipeline. Every number that later notebooks rely on (retained samples,
primary channels, flagged samples) is written to `derived/step0.json`.

Runtime: CPU, High-RAM is not needed. First run about 10 minutes (mostly the download); reruns skip what is cached.

In [ ]:
# Bootstrap: same cell in every notebook. On Colab: mount Drive, get the repo, install extras.
# The repo is private: add a read-only GitHub token to Colab Secrets as GH_TOKEN.
import os, sys, subprocess, base64
REPO_URL = "https://github.com/sanjanasinghx/dream2image-es606.git"
if "google.colab" in sys.modules and not os.environ.get("D2I_HEADLESS"):  # headless = run by d2i.runner
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO = "/content/dream2image-es606"
    try:
        auth = base64.b64encode(f"x-access-token:{userdata.get('GH_TOKEN')}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]  # token never written to disk
    except Exception:
        git = ["git"]  # no secret: works once the repo is public
    if not os.path.exists(REPO):
        subprocess.run([*git, "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run([*git, "-C", REPO, "pull", "-q"], check=True)
    freeze = lambda: subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    before = freeze()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
    if freeze() != before:  # e.g. numpy upgraded under an already-loaded copy -> broken imports
        print("Packages changed: restarting the runtime. When it reconnects, Run all again.")
        os.kill(os.getpid(), 9)
else:
    REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from d2i import env
cfg = env.load_config()
rng = env.seed_everything(cfg["seed"])
print("data root:", env.data_root())

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from d2i import data, checks
tables = env.path(cfg, "results") / "tables"; tables.mkdir(exist_ok=True)
figs = env.path(cfg, "results") / "figures"; figs.mkdir(exist_ok=True)
load = lambda sid, col=None: data.load_eeg(cfg, sid, col)

## 1. Download and cache (resumable)

In [ ]:
shards = data.download(cfg)
print([s.name for s in shards])
n = data.cache_rows(cfg, shards)
metas = data.load_meta(cfg)
assert len(metas) == cfg["dataset"]["expected_n_rows"], f"{len(metas)} rows cached"

## 2. Is every EEG column z-scored on its own?
If yes, absolute power is gone: only spectral *shape* can be compared across samples.

In [ ]:
scaling = checks.scaling_table(cfg, metas, load)
scaling.to_csv(tables / "scaling_check.csv", index=False)
live = scaling[scaling.sd > cfg["dataset"]["dead_sd_floor"]]
print(live.groupby("source")[["is_z"]].mean().rename(columns={"is_z": "frac_live_channels_z_scored"}))

## 3. Are the 15/30/60 s windows the tail of the 120 s recording?

In [ ]:
nest = checks.nesting_table(metas)
nest.to_csv(tables / "nesting_check.csv", index=False)
counts = nest.groupby(["window_s", "verdict"]).size().unstack(fill_value=0)
print(counts)
mis_frac = (nest.verdict == "misaligned").mean()
assert mis_frac <= cfg["checks"]["misaligned_stop_fraction"], f"{mis_frac:.1%} misaligned: stop and inspect"

## 4. Participants, sites and exclusions

In [ ]:
parts = checks.participants_table(metas, data.parse_title)
parts.to_csv(tables / "participants.csv", index=False)
excl = checks.exclusions_table(cfg, metas, nest, parts)
excl.to_csv(tables / "sample_exclusions.csv", index=False)
ret = excl[excl.retained]
print(f"participants: {parts.participant.nunique()} (paper: {cfg['participant']['expected_n']})")
print(f"retained {len(ret)}/{len(excl)} samples from {ret.participant.nunique()} participants")
print(ret.groupby("site").agg(samples=("sample_id", "size"), participants=("participant", "nunique")))
print("awakening tags (subject site):", ret.tag_class.fillna("none").value_counts().to_dict())
print("excluded:\n", excl.loc[~excl.retained, ["sample_id", "title", "reasons"]].to_string(index=False))
assert parts.participant.nunique() == cfg["participant"]["expected_n"]

## 5. Primary channel set
Channels live in at least 95% of retained samples (Cz is dead in every `subject`-site recording).

In [ ]:
avail = checks.channel_availability(cfg, scaling, excl)
avail.to_csv(tables / "channel_availability.csv", index=False)
primary = avail.loc[avail.primary, "channel"].tolist()
print(avail.to_string(index=False))
print("primary:", primary)

## 6. Is the undocumented channel order spatially plausible?

In [ ]:
order_res, corr = checks.channel_order_check(cfg, metas, load, rng)
corr.to_csv(tables / "channel_corr_mean.csv")
print(order_res)

## 7. Site differences in the spectrum
If the sites separate completely above 45 Hz, anything above ~30 Hz is a site marker, not brain signal.

In [ ]:
hf, f, spectra = checks.site_spectra(cfg, metas, parts, load)
hf.to_csv(tables / "site_hf_fraction.csv", index=False)
by = hf.groupby("site")["log10_frac_45_100"].agg(["count", "min", "median", "max"])
print(by)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
colors = {"case": "#c0504d", "subject": "#4a6fa5"}
m = f <= 100
for s, c in colors.items():
    sel = (hf.site == s).values
    ax[0].hist(hf.log10_frac_45_100[sel], bins=25, alpha=.7, color=c, label=f"{s} (n={sel.sum()})")
    ax[1].plot(f[m], spectra[sel][:, m].T, color=c, alpha=.08, lw=.6)
    ax[1].plot(f[m], spectra[sel][:, m].mean(0), color=c, lw=2, label=s)
ax[0].set(xlabel="log10 P(45-100 Hz) / P(1-45 Hz)", ylabel="samples", title="High-frequency power by site")
ax[1].set(xlabel="Hz", ylabel="log10 PSD (z-units²/Hz)", title="Channel-mean spectra, eeg_120s")
for a in ax: a.legend()
fig.tight_layout(); fig.savefig(figs / "site_spectra.pdf"); fig.savefig(figs / "site_spectra.png", dpi=150)

## 8. Per-epoch scale (sensitivity flags)

In [ ]:
so = checks.scaling_outliers(cfg, excl, load)
so.to_csv(tables / "scaling_outliers.csv", index=False)
flagged = so.loc[so.retained & so.scaling_flag, "sample_id"].tolist()
print("flagged for sensitivity analysis:", flagged)
print("samples with full 120 s (E1 available):", int((so.retained & (so.len_s >= 120)).sum()))

## 9. Save the decisions later notebooks use

In [ ]:
step0 = {
    "n_rows": len(metas),
    "retained": ret.sample_id.tolist(),
    "n_participants_retained": int(ret.participant.nunique()),
    "primary_channels": primary,
    "scaling_flagged": flagged,
    "nesting_counts": {str(k): v for k, v in counts.to_dict().items()},
    "channel_order": order_res,
    "site_hf": by.round(3).to_dict(orient="index"),
}
out = env.path(cfg, "derived") / "step0.json"
out.write_text(json.dumps(step0, indent=2, default=str))
env.write_manifest(cfg, "01_data", [out, *sorted(tables.glob("*.csv"))])
print(json.dumps({k: v for k, v in step0.items() if k != "retained"}, indent=1, default=str))